# Cloud Pendulum benchmark

This notebook runs the same benchmark as `benchmark.py`, one step at a time, and plots the results at the end.

What happens:
1. **Configuration**: pick the cells, the experiment length and the controller.
2. **Disturbances**: define the disturbances the system applies during the run.
3. **Run**: run one experiment on each cell, recording the state, torque and controller output at every control step.
4. **Save**: write `params.json`, `result.json`, `data.json` and the videos to `output/benchmark_<TIMESTAMP>/`. This is the same layout `benchmark.py` uses, so `plot.py` works on either.
5. **Plot**: show a score summary and the per-iteration time series, and play the videos.

> Running section 3 uses the **real hardware**. Each cell takes about `experiment_time + preparation_time` seconds, plus time to download the video.

In [ ]:
import math
import os
import json
import time
from datetime import datetime

import matplotlib.pyplot as plt
from IPython.display import Video, display

from pycloudpendulum import StateDependentPositionDisturbance, StateDependentTorqueDisturbance
from pycloudpendulumbenchmark import ControlLoop
from pycloudpendulumbenchmark.evaluators import GoalHeightEvaluator

from controller.pid_controller import PointPIDController
from video import download_video

%matplotlib inline

## 1. Configuration

- `user_token`: your Cloud Pendulum user token. **Set this before running.** Don't commit a real token.
- `cell_ids`: the hardware cells to run on. Each cell runs one iteration.
- `experiment_time`: length of each experiment, in seconds.
- `preparation_time`: time the cell gets to reach its initial state before control starts.
- `delta_time`: control period in seconds (0.002 s = 500 Hz).
- `EvaluatorType`: computes the score. `GoalHeightEvaluator` rewards time spent with the pendulum tip above the goal height.
- `ControllerType`: the controller under test. See `controller/pid_controller.py`.

In [ ]:
user_token = "USER TOKEN"
cell_ids = [
    201, 202, 203, 204,
]
experiment_time = 30.0
experiment_type = "DoublePendulum"
delta_time = 0.002
num_actuators = 2
preparation_time = 5.0
EvaluatorType = GoalHeightEvaluator
ControllerType = PointPIDController

### The controller interface

To work with the benchmarking interface, a controller class **must inherit from `ControllerBase`** (from `pycloudpendulumbenchmark`). `ControllerBase` is an abstract base class, so a subclass has to implement both of its abstract methods:

- `__init__(self)`: takes no arguments. The benchmark creates a **new instance for each cell** with `ControllerType()`, so set gains, limits, the goal and any internal state (such as integrator sums or error history) here. That way each run starts clean.
- `get_control_output(self, state: list[MotorState]) -> list[float]`: called once per control step (every `delta_time` seconds). `state` has one `MotorState` per actuator, each with `.position` (rad) and `.velocity` (rad/s). Return one torque command per actuator.

Leave either method out and Python raises a `TypeError` when the class is instantiated. A minimal example:

```python
from pycloudpendulumbenchmark import ControllerBase, MotorState

class ZeroController(ControllerBase):
    def __init__(self):
        pass

    def get_control_output(self, state: list[MotorState]) -> list[float]:
        return [0.0, 0.0]
```

`PointPIDController` in `controller/pid_controller.py` follows this pattern. To benchmark a different controller, write a `ControllerBase` subclass and set `ControllerType` to it. The next cell checks that the chosen class meets the interface before any hardware time is used.

In [ ]:
from pycloudpendulumbenchmark import ControllerBase

assert issubclass(ControllerType, ControllerBase), \
    f"{ControllerType.__name__} must inherit from pycloudpendulumbenchmark.ControllerBase"
ControllerType()  # raises TypeError if an abstract method is not implemented
print(f"{ControllerType.__name__} implements the ControllerBase interface")

### Control loop with a preparation time

`ControlLoop.start()` in `pycloudpendulumbenchmark` hardcodes `preparation_time=0.0`, so this subclass passes the configured value through. It also adds helpers that read the measured torque and fetch the experiment logs, which include the disturbance start and end messages the plots use.

In [ ]:
class BenchmarkControlLoop(ControlLoop):
    """ControlLoop with a configurable preparation time."""

    def __init__(self, *args, preparation_time: float = 0.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.preparation_time = preparation_time

    def start(self):
        session_token, _ = self.client.start_experiment(
            user_token=self.user_token,
            experiment_type=self.experiment_type,
            experiment_time=self.experiment_time,
            preparation_time=self.preparation_time,
            initial_position=self.initial_position,
            disturbances=self.disturbances,
            record=True,
            cell_id=self.cell_id,
        )
        self.i = 0
        self.loop_start_time = time.time()
        self.measured_time = 0
        self.session_token = session_token

    def get_torque(self) -> list[float]:
        return self.client.get_torque(self.session_token)

    def get_logs(self) -> list[str]:
        return self.client.get_logs(self.session_token)

## 2. Disturbances

Disturbances are *state dependent*. Each one waits until the pendulum has stayed near `goal_position` (within `target_pos_threshold` / `target_vel_threshold`) for `wait_at_target_seconds`, and then fires. Disturbances fire in the order listed.

1. **Position disturbances (24)**: once the pendulum is upright (`[π, 0]`), move it to a new configuration. That is 8 inner-joint angles (0 to 7π/4 in steps of π/4) × 3 outer-joint angles (0, −π/2, π/2). The controller must swing it back up.
2. **Torque pulses at the top (10)**: once upright, apply a 0.1 s torque pulse to the inner joint, from ±0.02 to ±0.1 Nm.
3. **Small pushes at the bottom (20)**: with the pendulum hanging (`[0, 0]`, with a very loose position threshold), apply repeated 0.01 Nm pulses.

In [ ]:
disturbances = []

# 1. Position disturbances from the upright position
for i in range(8 * 3):
    i_inner = i // 3
    i_outer = i % 3
    p_inner = float(i_inner) * math.pi / 4.0
    p_outer = [0.0, -math.pi / 2.0, math.pi / 2.0][i_outer]
    disturbances.append(StateDependentPositionDisturbance(
        wait_at_target_seconds=3.0,
        goal_position=[math.pi, 0.0],
        disturbance_position=[p_inner, p_outer],
        target_pos_threshold=0.3,
        target_vel_threshold=50.0,
    ))

# 2. Torque pulses on the inner joint while upright
for t in [0.02, 0.04, 0.06, 0.08, 0.1, -0.02, -0.04, -0.06, -0.08, -0.1]:
    disturbances.append(StateDependentTorqueDisturbance(
        wait_at_target_seconds=3.0,
        goal_position=[math.pi, 0.0],
        torque=[t, 0.0],
        duration=0.1,
        target_pos_threshold=0.3,
        target_vel_threshold=50.0,
    ))

# 3. Small pushes near the hanging position
for i in range(20):
    disturbances.append(StateDependentTorqueDisturbance(
        wait_at_target_seconds=0.2,
        goal_position=[0.0, 0.0],
        torque=[0.01, 0.0],
        duration=0.1,
        target_pos_threshold=1000.0,
        target_vel_threshold=50.0,
    ))

print(len(disturbances), "disturbances defined")

## 3. Run the benchmark

Make the output directory and save the parameters. The token is left out of `params.json` here so it doesn't end up in output files you might share.

In [ ]:
run_name = "benchmark_" + datetime.now().strftime("%Y-%m-%d-%H:%M:%S")
out_dir = "output/" + run_name
os.makedirs(out_dir, exist_ok=True)
with open(out_dir + "/params.json", "w") as f:
    json.dump({
        "cell_ids": cell_ids,
        "experiment_time": experiment_time,
        "experiment_type": experiment_type,
        "disturbances": str(disturbances),
    }, f)
print("Saving results to", out_dir)

For each cell:
- Build a fresh controller and evaluator, then start the experiment.
- At every control step, read the state and the measured torque, compute the controller output, update the score and log everything. Then send the command with `control_loop.step()`, which also keeps the loop on its `delta_time` schedule.
- Afterwards, fetch the logs, stop the session and download the recorded video.

If one cell fails (bad token, cell busy, connection error and so on), the exception is saved in its result and the next cell still runs.

In [ ]:
iteration_data = []
results = []

for iteration, cell_id in enumerate(cell_ids):
    position_history = [[] for _ in range(num_actuators)]
    velocity_history = [[] for _ in range(num_actuators)]
    torque_history = [[] for _ in range(num_actuators)]
    controller_output_history = [[] for _ in range(num_actuators)]
    time_history = []

    result = {"cell_id": cell_id, "exception": None}
    results.append(result)

    controller = ControllerType()
    evaluator = EvaluatorType()
    control_loop = BenchmarkControlLoop(
        user_token=user_token,
        experiment_time=experiment_time,
        experiment_type=experiment_type,
        dt=delta_time,
        cell_id=cell_id,
        initial_position=None,
        disturbances=disturbances,
        preparation_time=preparation_time,
    )
    print(f"[{iteration + 1}/{len(cell_ids)}] Running on cell {cell_id} ...")
    try:
        control_loop.start()

        while not control_loop.finished():
            state = control_loop.get_state()
            torque = control_loop.get_torque()
            control_loop_time = control_loop.time()

            controller_output = controller.get_control_output(state)
            evaluator.evaluate(state, controller_output, control_loop_time)

            for i in range(num_actuators):
                position_history[i].append(float(state[i].position))
                velocity_history[i].append(float(state[i].velocity))
                torque_history[i].append(float(torque[i]))
            for i in range(len(controller_output)):
                controller_output_history[i].append(float(controller_output[i]))
            time_history.append(control_loop_time)

            control_loop.step(controller_output)

        result["result"] = "SUCCESS"
    except Exception as e:
        result["result"] = "FAILED"
        result["exception"] = str(e)
    finally:
        result["score"] = evaluator.get_score()

    logs = None
    video_url = None
    if control_loop.session_token is not None:
        try:
            logs = control_loop.get_logs()
        except Exception as e:
            print("Failed to fetch logs:", str(e))
        video_url = control_loop.stop()
    result["logs"] = logs
    result["video"] = download_video(out_dir + "/videos", video_url, iteration)
    print(f"    {result['result']}  score={result['score']:.2f}  exception={result['exception']}")

    iteration_data.append({
        "position": position_history,
        "velocity": velocity_history,
        "torque": torque_history,
        "controller_output": controller_output_history,
        "time": time_history,
    })

total_score = sum(r["score"] for r in results) / float(len(results))
print("Total score:", total_score)

## 4. Save results

The same files `benchmark.py` writes. You can also plot them later from the command line with `python plot.py <run_name>`.

In [ ]:
with open(out_dir + "/result.json", "w") as f:
    json.dump({"iterations": results, "score": total_score}, f)

with open(out_dir + "/data.json", "w") as f:
    json.dump({"iterations": iteration_data}, f)

print("Saved", out_dir)

## 5. Plots

To plot an earlier run instead of the one above, set `run_name` in the next cell, for example `run_name = "benchmark_2026-11-22-18:41:42"`. Then run the cells below. They only read from `output/<run_name>/`.

In [ ]:
# run_name = "benchmark_2026-11-22-18:41:42"  # uncomment to plot an earlier run
with open(f"output/{run_name}/result.json") as f:
    saved_result = json.load(f)

### Score per cell

Failed cells are shown in grey. The dashed line is the average score.

In [ ]:
its = saved_result["iterations"]
labels = [str(it["cell_id"]) for it in its]
scores = [it["score"] for it in its]
colors = ["tab:blue" if it["result"] == "SUCCESS" else "tab:gray" for it in its]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(labels, scores, color=colors)
ax.axhline(saved_result["score"], color="tab:red", linestyle="--",
           label=f"Average: {saved_result['score']:.2f}")
for x, s in zip(labels, scores):
    ax.annotate(f"{s:.2f}", (x, s), ha="center", va="bottom")
ax.set_xlabel("Cell")
ax.set_ylabel("Score")
ax.set_title(run_name)
ax.spines[["top", "right"]].set_visible(False)
ax.legend()
plt.show()

for it in its:
    if it["exception"]:
        print(f"Cell {it['cell_id']}: {it['result']}, {it['exception']}")

### Time series per iteration

This runs `plot.py` on the run. Each iteration gets one figure with four panels:
- **Vertical position** of each joint (the sum of the joint angles, as −cos). The red dashed line is the goal height threshold.
- **Joint velocity**.
- **Measured joint torque**.
- **Controller output torque**.

Shaded areas show when each type of disturbance was active. The figures are also saved to `output/<run_name>/plots.pdf`.

In [ ]:
%run plot.py {run_name}

# Close any empty figure left over from %run so it isn't displayed
for n in plt.get_fignums():
    if not plt.figure(n).axes:
        plt.close(n)

### Videos

Recordings of each iteration, if the download succeeded.

In [ ]:
for i, it in enumerate(saved_result["iterations"]):
    if it.get("video"):
        print(f"Iteration {i}, cell {it['cell_id']}, score {it['score']:.2f}")
        display(Video(f"output/{run_name}/videos/{it['video']}", width=480, embed=False))